## Cell 0. API keys

Paste your keys between the quotes below and run this cell before anything
else. Leave a line as `""` to use whatever is already exported in the
environment instead.

**Two things worth knowing before you paste.** This notebook is regenerated by
`build_q1_nb.py`, which rewrites every cell from source -- so a key typed here
is lost on the next rebuild. And a key typed here is saved inside the `.ipynb`
file, where it can reach git or a shared copy. For a key you intend to keep,
put it in `fourarm/env/keys.local.env` instead, which this cell reads
automatically and which is gitignored and never regenerated.

In [ ]:
# --- Cell 0. API keys. Run first. -------------------------------------------
import os, pathlib

# PASTE BETWEEN THE QUOTES. Leave "" to fall back to the environment or to
# env/keys.local.env.
KEYS = {
    "OPENAI_API_KEY": "",
    "GEMINI_API_KEY": "",
    "ANTHROPIC_API_KEY": "",
}

# An EMPTY value must never be written into the environment. Assigning ""
# unconditionally would blank a key that is already exported correctly, and
# the failure -- a 401 from a variable that is set but empty -- reads nothing
# like "you left the placeholder alone".
for _name, _value in KEYS.items():
    if _value.strip():
        os.environ[_name] = _value.strip()

# The persistent alternative. Same KEY=value format as env/models.env, one
# per line, # for comments. Read only for names not already set, so anything
# pasted above and anything already exported both win over the file.
_here = pathlib.Path.cwd()
_root = next((c for c in [_here] + list(_here.parents)
              if (c / "out").is_dir() and (c / "experiments").is_dir()), None)
_local = _root / "env" / "keys.local.env" if _root else None
if _local and _local.exists():
    for _line in _local.read_text().splitlines():
        _line = _line.strip()
        if not _line or _line.startswith("#") or "=" not in _line:
            continue
        _k, _, _v = _line.partition("=")
        _k, _v = _k.strip(), _v.strip().strip("\'\"")
        if _v and not os.environ.get(_k):
            os.environ[_k] = _v

# Report presence, NEVER the value. Printing a key would write it into the
# notebook's saved output, which is the same leak as pasting it into a cell
# and is easier to do by accident.
#
# The report loops over KEYS, so EVERY key the notebook can use needs a row
# there even when it is only ever supplied by keys.local.env. A name missing
# from KEYS still loads from the file, but silently, and a key that loads
# without being reported is indistinguishable from one that did not load.
for _name in KEYS:
    _set = bool(os.environ.get(_name))
    print("%-18s %s" % (_name, "set" if _set else "NOT SET"))
if _local:
    print("%-18s %s" % ("keys.local.env",
                        "read" if _local.exists() else "absent (optional)"))

# Experiment 2, Q1: Derivation

**When the text omits the capability-relevant quantity, can the model obtain it
from the scene?**

Read at rung **N0** only. The other rungs belong to Q3.

Every cell is independently runnable and idempotent. No cell overwrites a paid
run: the runners resume into their output file and skip trials already
answered. Cells that spend money print the call count and refuse to proceed
until `CONFIRM_SPEND` is set to that exact number.

Run this notebook with the working directory set to `fourarm/`, or anywhere
below it -- cell 1 finds the root itself.

In [ ]:
# --- Cell 1. Setup. No model calls. -----------------------------------------
import collections, csv, datetime, hashlib, json, math, os, pathlib, sys

# Find the package root: the directory holding out/ and experiments/.
here = pathlib.Path.cwd()
ROOT = None
for cand in [here] + list(here.parents):
    if (cand / "out").is_dir() and (cand / "experiments").is_dir():
        ROOT = cand
        break
if ROOT is None:
    raise SystemExit("run this from fourarm/ or below: no out/ + experiments/ found")
for p in (str(ROOT), str(ROOT / "ycb")):
    if p not in sys.path:
        sys.path.insert(0, p)

# RE-IMPORT, never reuse. Python caches modules in sys.modules, so running
# this cell a second time in a live kernel keeps whatever was on disk the
# FIRST time it ran. While the ex2 modules are being edited alongside the
# notebook that is a trap: the kernel holds the old vocabulary, and the
# failure surfaces cells later as a design-check assertion naming a face
# that no longer exists, which reads like a code error and is not one.
#
# Dropping the entries and importing fresh is used rather than
# importlib.reload because these modules import each other, and reload
# leaves a half-updated graph unless the order is exactly right.
for _stale in [m for m in list(sys.modules)
               if m.startswith(("experiments.ex2", "analysis.ex2"))
               or m in ("ycb_objects",)]:
    del sys.modules[_stale]

# WHICH KERNEL THIS IS, checked before the first project import.
#
# The very next line reaches core.decision.state_builder through
# mancheck -> vlm_allocator, and that imports numpy; visibility.py, in cell
# 3, needs PIL and scipy. On a kernel without them the notebook dies forty
# lines deep inside somebody else's module with "No module named 'numpy'",
# which reads as a broken repository rather than as a kernel picked from a
# list of six. Checked here, where the answer is one sentence.
_missing = []
for _m in ("numpy", "PIL", "scipy"):
    try:
        __import__(_m)
    except ImportError:
        _missing.append(_m)
if _missing:
    _venv = ROOT.parent / ".venv" / "bin" / "python"
    raise SystemExit(
        "WRONG KERNEL.\n"
        "  This kernel is  %s\n"
        "  and it has no %s.\n"
        "  Use instead     %s\n"
        "  In VS Code: Select Kernel, then Python Environments, then the\n"
        "  interpreter at that path. It is the only one in this tree with\n"
        "  ipykernel AND numpy, PIL and scipy. Several unrelated kernels are\n"
        "  registered on this machine and any of them will get this far and\n"
        "  then fail."
        % (sys.executable, ", ".join(_missing), _venv))

from core.cell import cell_config as C
from core.decision import model_registry as MR
from experiments.ex2 import grade as G
from experiments.ex2 import labels as L
from experiments.ex2 import mancheck as MC
from experiments.ex2 import prompts as P
from experiments.ex2 import run as R
from experiments.ex2 import solo as S
from experiments.ex2 import transforms as T
from experiments.ex2 import visibility as VIS
from analysis.ex2.ex2_stats import newcombe, paired_mean_ci, spans_zero, wilson
# The notebook machinery: loaders, the share definition, the paired
# contrast and the spend gate. In a module rather than in this cell so
# that Q2 and Q3 use the same ones rather than a second copy, and so
# that harness/h_ex2_q_common.py can pin them. What stays in the cells
# is what is a DECISION: the models, the rung, the conditions, the
# usable rule, each cost, and every CONFIRM_SPEND.
from analysis.ex2.ex2_q_common import (Outputs, answered,       # noqa
                                       coupling, fmt, full_flip_count,
                                       is_franka, keep_analysable,
                                       load_run, paired_diffs, pct,
                                       provenance_row, run_meta,
                                       sha256, share_at, share_counts,
                                       show, spend_gate)

# --- paths ------------------------------------------------------------------
CAPTURES = ROOT / "out" / "ex2_capture_block"
RUNS     = ROOT / "runs"
TABLES   = ROOT / "tables" / "ex2_q1"
FIGURES  = ROOT / "figures" / "ex2_q1"
for d in (RUNS, TABLES, FIGURES):
    d.mkdir(parents=True, exist_ok=True)

# --- constants, every one read from a source of truth ------------------------
RUNG        = "N0"                       # Q1 is read here and nowhere else
PREFERENCE  = "franka"
CONDITIONS  = ("congruent", "dims")
REPEATS     = 3
FACES       = P.RESTING_FACES            # small_face, large_face
LABEL       = "ycb_block"

FRANKA_MAX  = C.ARM_TYPES["franka"]["max_grasp_m"]
UR_MAX      = C.ARM_TYPES["ur10"]["max_grasp_m"]
DIMS        = T.DIMS_M[LABEL]
FACTS       = T.POSE_FACTS_BY_LABEL[LABEL]

# The registry has no hardcoded model list: aliases() reads FOURARM_MODELS.
try:
    ALIASES = MR.aliases()
except Exception as exc:
    ALIASES = []
    print("model registry unavailable (%s); set MODELS by hand below" % exc)
# THREE models since 2026-08-27. claude-sonnet-5 was added because the
# design needs a third model that CLEARS the two-way face probe: with two
# models, a single failure at cell 5b leaves one, and one model cannot show
# that a result is a property of models rather than of this one model.
# It is not here for being the most capable available; see env/models.env.
#
# claude_md RATHER THAN claude. Same model, claude-sonnet-5, at effort
# medium instead of the API default of high. At the default it read the
# two-way face probe at 65 percent against gpt's 95 and gemini's 100, and
# it failed by BIAS rather than blindness: large_face on 75 percent of
# trials, 90 percent right when the block lies flat and 40 percent when it
# stands. Deliberation is how a prior like "blocks lie flat" gains weight,
# so lower effort is the move that fits the failure. Cell 5b is what tests
# it. The default-effort runs stay on disk under the alias "claude".
#
# gpt_hi RATHER THAN gpt. Same model, gpt-5.6-terra, at reasoning_effort
# high instead of low. Claude runs at the Anthropic default effort of high
# and Gemini Flash exposes no effort control at all, so gpt at low made the
# one model with the LEAST test-time compute the yardstick for the other
# two. Effort is still not matched across providers and cannot be -- that
# stays in Limitations -- but the reasoning models are now on the same
# nominal tier.
#
# The low-effort runs are NOT deleted. runs/ex2_q1_cue2way_gpt_r*.jsonl
# record gpt at reasoning_effort low over this same sample and stay on disk
# as the evidence for what effort was worth here: 95 percent at low. A
# separate alias rather than an edit to GPT_PARAMS is what makes those rows
# still readable, which is the reason env/models.env gives for gpt_hi
# existing at all.
#
# Named rather than taken wholesale from ALIASES. FOURARM_MODELS also lists
# qwen and gpt_hi, and a run's model set must be a decision recorded here,
# not whatever the registry happens to carry. The fallback keeps the same
# three so a registry failure cannot silently shrink the design.
_WANT = ("gpt_hi", "gemini", "claude_md")
MODELS = tuple(a for a in ALIASES if a in _WANT) or _WANT
if set(MODELS) != set(_WANT):
    print("WARNING: %s requested, %s available from the registry. Every "
          "table below is per model, so a missing one narrows the design "
          "rather than breaking it -- but say so in the chapter."
          % (list(_WANT), list(MODELS)))

# --- credentials: reported, not assumed -------------------------------------
# Until 2026-08-27 a hand-added launcher cell started JupyterLab in a browser
# and refused to launch when a key was missing. That cell is gone: it spawned
# a NEW server every time it ran, which is how eight of them accumulated, each
# serving its own in-memory copy of this notebook, so an edit on disk could be
# invisible in the tab you were typing in. VS Code runs the kernel directly
# and needs no launcher -- but the key check it performed was worth keeping,
# so it lives here.
#
# This REPORTS rather than raises. Cells 1-5 and every analysis cell make no
# model calls and must stay runnable with no key at all. What it buys is
# learning about a missing key now instead of at cell 6, part-way into a run.
#
# The usual cause is launching VS Code from Finder or the Dock, which does not
# inherit a login shell, so a key exported in .zshrc is absent here while
# present in any terminal. The message below says so, because the symptom
# otherwise looks like a broken registry.
#
# key_var is read from the registry, never hardcoded: models.env lets each
# alias name its own variable, and a hardcoded OPENAI_API_KEY would check the
# wrong one the moment that is used.
MISSING_KEYS = []
for _alias in MODELS:
    try:
        _var = MR.describe(_alias)["key_var"]
    except Exception as _exc:
        MISSING_KEYS.append("%s: %s" % (_alias, _exc))
        continue
    if not os.environ.get(_var):
        MISSING_KEYS.append("%s: %s is not set" % (_alias, _var))

# Output paths travel together in one object, so a notebook cannot end up
# with a root and a tables directory that disagree. Rebound to bare names
# because every call site below reads better as write_csv(...) than as
# OUT.write_csv(...), and because leaving those call sites untouched is
# what made this extraction verifiable against the tables already on disk.
OUT = Outputs(ROOT, TABLES, FIGURES)
rel, write_csv = OUT.rel, OUT.write_csv

print("root        ", ROOT)
print("captures    ", CAPTURES.relative_to(ROOT), "(exists:", CAPTURES.is_dir(), ")")
print("rung        ", RUNG, " preference", PREFERENCE, " repeats", REPEATS)
print("models      ", MODELS, " (registry knows: %s)" % (ALIASES or "nothing"))
print("prompt ver  ", P.EX2_PROMPT_VERSION)
# Printed, not assumed. If a stale kernel ever slips past the re-import
# above, this is the line that shows it, at the top of the run rather than
# in an assertion twenty cells later.
print("faces       ", FACES, " chance %.1f%%" % (100.0 / len(FACES)))
if MISSING_KEYS:
    print("api keys     MISSING -- analysis runs, model calls will not:")
    for _m in MISSING_KEYS:
        print("               ", _m)
    print("             launch VS Code from a shell that exports them:")
    print("               open -a 'Visual Studio Code' <repo>")
else:
    print("api keys     present for %s" % (", ".join(MODELS),))
print()
print("block           %.3f x %.3f x %.3f m"
      % (DIMS["height"], DIMS["width"], DIMS["depth"]))
print("franka opens to  %.3f m   ur opens to %.3f m" % (FRANKA_MAX, UR_MAX))
print("resting faces   %s" % (FACES,))
for f in FACES:
    print("   %-11s needs %.3f m" % (f, FACTS[f]["grasp_m"]))

## Cell 2. Design check

No model calls. Derives the opening for each resting face from the authored
cuboid dimensions and asserts it matches what `transforms` declares. The prompt
states the bounding-box convention, so a divergence here would make the prompt
wrong rather than silent.

In [ ]:
# --- Cell 2. Design check. No model calls. ----------------------------------
from ycb_objects import YCB as _SPECS      # the authored object dictionary

# STALE-IMPORT GUARD. Cell 1 purges sys.modules before importing, so a
# module edited on disk is picked up whenever cell 1 is re-run. This
# catches the case where cell 1 was NOT re-run -- editing a module and
# jumping straight back to this cell -- and the worse case where the
# NOTEBOOK ITSELF is stale, because JupyterLab holds its own copy in the
# browser and does not re-read the file when it changes underneath. A
# stale cell 1 has no purge, so the modules stay old and the design
# assertion below fails naming a face that no longer exists. That reads
# like a code error and is not one, which is why this checks first and
# says which of the two it is.
#
# The comparison is against the SOURCE ON DISK, not against a constant
# written here, so it stays true across future vocabulary changes.
import re                                   # local: a stale Cell 1 may
                                            # not have imported it
_src = pathlib.Path(P.__file__).read_text()
_on_disk = re.search(r'EX2_PROMPT_VERSION\s*=\s*["\'](.+?)["\']', _src)
if _on_disk and _on_disk.group(1) != P.EX2_PROMPT_VERSION:
    raise SystemExit(
        "STALE IMPORT: this kernel holds prompts.py version %s, but the file "
        "on disk is %s.\n"
        "  Loaded faces: %s\n"
        "  Fix: re-run Cell 1, which drops the cached modules and imports "
        "fresh.\n"
        "  If re-running Cell 1 does not clear it, the NOTEBOOK is stale, not "
        "the kernel:\n"
        "  JupyterLab is running the copy it loaded into the browser. Use "
        "File > Reload\n"
        "  Notebook from Disk, then Restart Kernel and Run All."
        % (P.EX2_PROMPT_VERSION, _on_disk.group(1), list(FACES)))

# Each block prim is spawned already resting on a face, with size PRE-ORIENTED
# to that pose: size is (x, y, z) with z vertical. So the two horizontal
# extents are size[0] and size[1], and the opening is the smaller of them.
# YCB is keyed without the "ycb_" scene prefix.
PRIM_OF_FACE = {L.TRUE_POSE[p]: p for p, lab in L.POSE_ENTRIES.items()
                if lab == LABEL}

design_rows = []
problems = []
for face in FACES:
    prim = PRIM_OF_FACE[face]
    size = _SPECS[prim.replace("ycb_", "")]["size"]
    horiz = sorted(size[:2], reverse=True)          # a = larger, b = smaller
    vertical = size[2]
    opening = min(horiz)
    declared = FACTS[face]["grasp_m"]

    if abs(opening - declared) > 1e-9:
        problems.append("%s: bounding box gives %.3f, transforms declares %.3f"
                        % (face, opening, declared))
    # A real permutation check, all three extents. It compared only the
    # SMALLEST until 2026-08-27, so a prim sized 0.200 x 0.200 x 0.050 --
    # not the block at all -- passed a check whose message said it was
    # verifying a permutation. That matters more with two faces than it
    # did with three: there are fewer cross-checks left, and this cell is
    # what stands between a mis-authored prim and the whole experiment.
    if (sorted(round(v, 6) for v in list(horiz) + [vertical])
            != sorted(round(DIMS[k], 6) for k in ("height", "width", "depth"))):
        problems.append(
            "%s: extents %s are not a permutation of the block %s"
            % (face, sorted(list(horiz) + [vertical]),
               sorted(DIMS[k] for k in ("height", "width", "depth"))))

    franka_ok = declared <= FRANKA_MAX
    ur_ok = declared <= UR_MAX
    design_rows.append([face, "%.3f" % vertical, "%.3f" % horiz[0],
                        "%.3f" % horiz[1], "%.3f" % declared,
                        "%.3f" % FRANKA_MAX, "%.3f" % UR_MAX,
                        franka_ok, ur_ok,
                        "franka, preference satisfied" if franka_ok
                        else "UR, preference overridden"])

# The design only works if the Franka is feasible on one face and not the
# other, and the UR on both. Anything else and Q1 has no contrast.
feasible = [r[0] for r in design_rows if r[7]]
if sorted(feasible) != ["small_face"]:
    problems.append("franka feasible on %s, expected small_face alone"
                    % sorted(feasible))
if not all(r[8] for r in design_rows):
    problems.append("a UR is infeasible somewhere; it must be legal everywhere")

show(["face", "vert", "horiz_a", "horiz_b", "opening", "franka", "ur", "picks"],
     [[r[0], r[1], r[2], r[3], r[4], r[7], r[8], r[9]] for r in design_rows])
print()
if problems:
    raise AssertionError("DESIGN CHECK FAILED:\n  " + "\n  ".join(problems))
print("PASS  every opening is the smaller horizontal extent, and the Franka")
print("      is feasible on small_face and not on large_face.")
print("      A third face, the middle one, was withdrawn on 2026-08-27: it")
print("      was flat like large_face and differed only in geometry, which")
print("      made it the sharper test, but no model read it (GPT 58%,")
print("      Fisher p=0.76 over 81 trials). The cost is that a model")
print("      reading posture and applying a rule can no longer be told")
print("      apart from one deriving the opening from geometry.")

write_csv("tab_ex2_q1_design.csv",
          ["resting_face", "vertical_m", "horiz_a_m", "horiz_b_m",
           "opening_needed_m", "franka_max_m", "ur_max_m", "franka_feasible",
           "ur_feasible", "deriving_model_picks"],
          design_rows)

## Cell 3. Capture inventory and legality

No model calls. Loads the captures, checks the grid is complete, re-asserts the
recorded settle heights, and runs the **real validator** over every scene to
establish which positions can carry the contrast at all.

This cell is the reason the sample is 29 positions rather than 30, and it fails
loudly rather than letting the analysis assume otherwise.

**Why the settle heights are re-checked here.** The prompt never says which flat
orientation to expect. The convention sentence -- *an object resting flat lies on
its largest face* -- was deliberately left out, because capture enforces it
instead: `capture_ex2_scene.py` fails a capture that settles at the wrong height
rather than relabelling it with the face it was asked for. That assertion is real
and it does raise, but it post-dates most of the captures on disk, and the trail
check below it compares only the recorded face *word* against the prim -- never
the height that word is supposed to describe. So the single guarantee standing
behind the prompt's silence was being taken on trust at the point where the data
is actually read. Every capture records `ex2.settled[name]`, so checking it costs
nothing. The tolerance is read out of the capture script's source rather than
typed here, so it cannot drift from the value the captures were accepted under.

In [ ]:
# --- Cell 3. Capture inventory and legality. No model calls. ----------------
import re                                   # local, as in Cell 2: Cell 1
                                            # does not import it, so this
                                            # cell must not depend on Cell 2
                                            # having been run first
scenes = R.load_scenes(str(CAPTURES))          # normalises the idle UR
raw    = R.load_scenes(str(CAPTURES), present_ur=False)   # as written
trail  = [json.loads(l) for l in open(CAPTURES / "consults.jsonl") if l.strip()]

by_pos = collections.defaultdict(dict)
for s in scenes:
    pos, member = s["seq"].rsplit("_", 1)
    prim = [o["name"] for o in s["state"]["objects"] if LABEL.split("_")[-1] in o["name"]][0]
    by_pos[pos][L.TRUE_POSE[prim]] = s

# DERIVED, never literal. This read "90 captures / 30 positions" until
# 2026-08-27 and raised the moment four positions were added to the capture
# plan. A count typed here goes stale silently; one derived from the
# directory cannot. What actually matters is not the total but that every
# position carries every face, which `missing` below checks.
inv_problems = []
if len(scenes) != len(by_pos) * len(FACES):
    inv_problems.append("expected %d captures (%d positions x %d faces), "
                        "found %d" % (len(by_pos) * len(FACES), len(by_pos),
                                      len(FACES), len(scenes)))
missing = {p: sorted(set(FACES) - set(v)) for p, v in by_pos.items()
           if set(v) != set(FACES)}
if missing:
    inv_problems.append("positions missing a face: %s" % missing)
if len({s["seq"] for s in scenes}) != len(scenes):
    inv_problems.append("duplicate seq ids")

# The face is derived from the PRIM, never from the trail's word, and the
# trail is then checked against it.
#
# Captures written before 2026-08-27 record ex2.resting_face in a superseded
# vocabulary where "upright" meant small_face and "small_face" meant the
# retired middle face. capture_ex2_scene.py now writes the geometric name
# directly, so new captures need no translation; this map reads the old ones
# and is why the check is against the prim rather than the word.
TRAIL_FACE = {"upright": "small_face", "small_face": "edge",
              "large_face": "large_face"}

# READ FROM THE CAPTURE SCRIPT, not typed here. capture_ex2_scene.py
# imports isaaclab at module scope and cannot be imported into this kernel,
# and a literal copied into the notebook would go stale the moment the
# tolerance is retuned -- which it was, from 0.010 to 0.005, on 2026-08-27.
# Same regex-the-source trick cell 2 uses for EX2_PROMPT_VERSION.
_cap_src = (ROOT / "ycb" / "capture_ex2_scene.py").read_text()
_tol = re.search(r"^SETTLE_TOL_M\s*=\s*([0-9.]+)", _cap_src, re.M)
if not _tol:
    raise SystemExit(
        "SETTLE_TOL_M not found in ycb/capture_ex2_scene.py. It is the "
        "tolerance the captures were accepted under and the notebook must "
        "not invent one; if it was renamed, update this cell.")
SETTLE_TOL_M = float(_tol.group(1))
for rec in trail:
    prim = [o["name"] for o in rec["state"]["objects"] if "block" in o["name"]][0]
    want = L.TRUE_POSE.get(prim)
    if want is None:
        continue          # a retired-face capture; load_scenes drops it too
    word = (rec.get("ex2") or {}).get("resting_face")
    # BOTH vocabularies are accepted, and only because each is checked
    # against the prim. A word is fine if it already IS the derived face
    # (written 2026-08-27 or later) or if it translates to it (written
    # before). Anything else is a genuine disagreement. Accepting both is
    # not laxity: the prim is the truth in either case, and the word is
    # never the thing consulted downstream.
    if word != want and TRAIL_FACE.get(word) != want:
        inv_problems.append("%s: trail says %r, prim says %r"
                            % (rec["seq"], word, want))

    # THE SETTLE HEIGHTS, RE-ASSERTED WHERE THE DATA IS READ.
    #
    # The prompt says nothing about which flat orientation to expect. The
    # convention sentence ("an object resting flat lies on its largest
    # face") was deliberately NOT added, on the grounds that capture
    # enforces it instead -- and it does: capture_ex2_scene.py raises on a
    # capture that settles at the wrong height rather than labelling it
    # with the face it was asked for. But that assertion post-dates most
    # captures on disk, and the check above compares only the trail's face
    # WORD against the prim, never the height that word is supposed to
    # describe. So the one guarantee standing behind the prompt's silence
    # was, at this point, taken on trust. It is not expensive to check.
    for name, d in (rec.get("ex2") or {}).get("settled", {}).items():
        z, want_z = d.get("z_above_table"), d.get("expected_rest_z")
        if want_z is None:
            inv_problems.append("%s: %s has no expected_rest_z, so its "
                                "resting face was never verified"
                                % (rec["seq"], name))
        elif abs(z - want_z) > SETTLE_TOL_M:
            inv_problems.append(
                "%s: %s settled at z=%.4f, expected %.4f within %.3f. It is "
                "not on the face this capture claims."
                % (rec["seq"], name, z, want_z, SETTLE_TOL_M))

print("captures %d   positions %d   faces per position %s"
      % (len(scenes), len(by_pos),
         sorted({len(v) for v in by_pos.values()})))
print("idle UR presented: %s"
      % dict(collections.Counter(s["idle_ur"] for s in scenes)))
print("idle UR as captured: %s"
      % dict(collections.Counter(
          tuple(sorted(a["name"] for a in s["state"]["arms"]
                       if a["state"] == "IDLE" and a["name"].startswith("ur")))
          for s in raw)))
print()

# --- the real validator, per scene ------------------------------------------
legal = {}
for pos in sorted(by_pos):
    for face, s in by_pos[pos].items():
        st, meta = T.transform({"state": s["state"],
                                "positions_exact": s["positions_exact"]},
                               "congruent")
        tid = R.flip_task_id(s["state"], meta["flip_prim"])
        legal[(pos, face)] = sorted(R.legal_arms(s, meta["flip_prim"], tid))

def has_franka(arms):
    return any(a.startswith("franka") for a in arms)

# TWO independent preconditions, not one. Legality asks whether the aperture
# contrast EXISTS at a position; visibility asks whether the block can be
# SEEN there. A position can carry the full contrast with the block hidden
# behind the Franka, and until 2026-08-27 nothing noticed: e10 presents 3%
# of the median block area and GPT inverted both its posture trials.
#
# visibility.verdict reads pixels only and never a model reply, so a
# position is never excluded for having scored badly.
#
# Run PER PREFIX, not over the directory at once. Each pose is scored
# against the median of its own pose, and the west and east banks sit at
# different distances from the camera: a w block renders about 10 percent
# larger than an e block in the same pose. One pooled median would raise
# the bar for the far bank and lower it for the near one, which is a
# comparison between banks rather than a test of occlusion.
OCCLUDED, _vis_detail = [], {}
for _pfx in sorted({p[0] for p in by_pos}):
    _c = VIS.measure(str(CAPTURES), prefix=_pfx)
    _ok, _bad, _d = VIS.verdict(_c)
    print("visibility, %s bank:" % _pfx)
    print(VIS.report(_d, _ok, _bad))
    print()
    OCCLUDED += _bad
    _vis_detail.update(_d)

USABLE, excluded = [], {}
for pos in sorted(by_pos):
    sm, lg = (legal[(pos, f)] for f in ("small_face", "large_face"))
    ok = has_franka(sm) and lg and not has_franka(lg)
    if ok:
        USABLE.append(pos)
    else:
        excluded[pos] = {"small_face": sm, "large_face": lg}

show(["position", "small_face", "large_face", "usable"],
     [[pos, ",".join(legal[(pos, "small_face")]) or "NONE",
       ",".join(legal[(pos, "large_face")]) or "NONE",
       "yes" if pos in USABLE else "NO"] for pos in sorted(by_pos)])
print()
USABLE = [p for p in USABLE if p not in OCCLUDED]
print("positions carrying the full contrast and showing the block: %d of %d"
      % (len(USABLE), len(by_pos)))
for pos in sorted(set(OCCLUDED)):
    print("  EXCLUDED %s  the block is occluded here (%.2f of the pose"
          % (pos, _vis_detail[pos]["worst"]))
    print("           median, worst in %s). The contrast may exist, but a"
          % _vis_detail[pos]["worst_pose"])
    print("           perception result from a picture that does not show")
    print("           the object is not a result about the model.")
for pos, v in excluded.items():
    print("  EXCLUDED %s  %s" % (pos, v))
    print("           the franka is never legal here, so there is no arm choice")
    print("           to make and no contrast to measure. Excluded with cause,")
    print("           not dropped silently.")

write_csv("tab_ex2_q1_inventory.csv",
          ["position", "usable", "occluded", "idle_ur", "legal_small_face",
           "legal_large_face"],
          [[pos, pos in USABLE, pos in OCCLUDED,
            by_pos[pos]["small_face"]["idle_ur"],
            ";".join(legal[(pos, "small_face")]),
            ";".join(legal[(pos, "large_face")])] for pos in sorted(by_pos)])

if inv_problems:
    raise AssertionError("INVENTORY FAILED:\n  " + "\n  ".join(inv_problems))
if len(USABLE) < 20:
    raise AssertionError("only %d usable positions; the contrast is not "
                         "estimable and the run should not be paid for"
                         % len(USABLE))

# --- what the PAID cells ask about ------------------------------------------
# Defined here, ONCE, and used by both cell 6 and cell 7. Putting the choice
# in each paid cell would let the two be set differently, so congruent and
# dims would cover different scene sets and the paired contrast in cell 10
# would silently compare two different samples.
#
# TRUE is the standing decision: ask about every captured scene, including
# the excluded positions, and drop them in cell 8 at ANALYSIS time. It costs
# a little more and buys something the write-up needs -- the excluded rows
# are in the data, so the exclusion can be shown to predate any accuracy
# result rather than being read as a position dropped for scoring badly.
#
# Set FALSE to pay only for the usable positions. The analysis is unaffected
# either way: cell 8 restricts to USABLE regardless.
RUN_ALL_POSITIONS = True

CALL_SCENES = ([s for s in scenes
                if s["seq"].rsplit("_", 1)[0] in USABLE]
               if not RUN_ALL_POSITIONS else scenes)

print()
print("PASS  inventory complete, %d positions usable." % len(USABLE))
print("      paid cells will ask about %d scenes (%s)"
      % (len(CALL_SCENES),
         "all captured, excluded positions included on purpose"
         if RUN_ALL_POSITIONS else "usable positions only"))

## Cell 4. Prompt inspection

No model calls. Renders the N0 prompt for both conditions and prints them in
full, so what the model reads is on the record beside the numbers it produced.

**dims changes two parts of the prompt, not just the state.** Withholding
`resting_face` and `opening_needed_m` means the field list must stop promising
them and R3 must stop pointing at one of them. R3 is *substituted*, not
deleted: deleting it would test the value of knowing the constraint exists,
which is not the question. A model reading a rule that names a missing field
could reasonably decide the rule is inapplicable, and that would look like a
derivation failure while being a rule-reading failure.

This cell prints both substitutions and asserts neither of them says where the
opening comes from (that is factor C) or calls the absence an error (that
would steer the model toward abstaining, and abstention is one of the
measures).

In [ ]:
# --- Cell 4. Prompt inspection. No model calls. -----------------------------
probe_scene = by_pos[USABLE[0]]["large_face"]

rendered = {}
for cond in CONDITIONS:
    msgs, meta = S.render(probe_scene, cond, PREFERENCE, RUNG)
    rendered[cond] = {
        "system": msgs[0]["content"],
        "user": [b for b in msgs[1]["content"] if b.get("type") == "text"][0]["text"],
        "meta": meta}

for cond in CONDITIONS:
    print("=" * 74)
    print("SYSTEM PROMPT  --  %s, rung %s" % (cond.upper(), RUNG))
    print("=" * 74)
    print(rendered[cond]["system"])
    print()

print("=" * 74)
print("DIFF, congruent -> dims  (system prompt)")
print("=" * 74)
import difflib
for line in difflib.unified_diff(rendered["congruent"]["system"].splitlines(),
                                 rendered["dims"]["system"].splitlines(),
                                 lineterm="", n=1):
    if line[:3] not in ("---", "+++", "@@ "):
        print(line)
print()

print("=" * 74)
print("USER MESSAGE  --  dims  (the state the model reads)")
print("=" * 74)
print(rendered["dims"]["user"])
print()

# --- what dims changes in the PROMPT, not just the state --------------------
head = lambda t: t[:t.index("\nYOUR ANSWER")]
r3_of = lambda t: t[t.index("R3  Gripper opening"):t.index("R4  Load")]

print("=" * 74)
print("R3, side by side")
print("=" * 74)
for cond in CONDITIONS:
    print("--- %s" % cond)
    print(r3_of(rendered[cond]["system"]).rstrip())
    print()

print("=" * 74)
print("THE OBJECT FIELD LIST, side by side")
print("=" * 74)
for cond in CONDITIONS:
    print("--- %s" % cond)
    print(P.CONDITIONS[cond]["object_fields"])
    print()

# --- assertions -------------------------------------------------------------
pp = []

# The STATE.
if '"resting_face"' not in rendered["congruent"]["user"]:
    pp.append("congruent must state the resting face")
if '"opening_needed_m"' not in rendered["congruent"]["user"]:
    pp.append("congruent must state the opening")
if '"resting_face"' in rendered["dims"]["user"]:
    pp.append("dims must WITHHOLD the resting face")
if '"opening_needed_m"' in rendered["dims"]["user"]:
    pp.append("dims must WITHHOLD the opening")
if '"size_upright_m"' not in rendered["dims"]["user"]:
    pp.append("dims must keep the object's own dimensions, or nothing is derivable")

# The PROMPT must follow the state. A field list that promises what the state
# does not carry, or a rule pointing at a field that is not there, makes the
# model solve a comprehension puzzle rather than the derivation under test.
dims_head = head(rendered["dims"]["system"])
for f in P.DIMS_WITHHELD:
    if ('"%s"' % f) in dims_head:
        pp.append("the dims field list still promises %s" % f)
    if ('"%s"' % f) not in head(rendered["congruent"]["system"]):
        pp.append("the congruent field list omits %s" % f)

dims_r3 = r3_of(rendered["dims"]["system"])
if "R3  Gripper opening" not in rendered["dims"]["system"]:
    pp.append("R3 must be SUBSTITUTED in dims, never deleted: deleting it "
              "would test the value of knowing the constraint exists")
if '"opening_needed_m"' in dims_r3:
    pp.append("R3 in dims still points at the withheld field")
if "opening_max_m" not in dims_r3:
    pp.append("R3 in dims dropped the capability check itself")
for cond in ("congruent",):
    if '"opening_needed_m"' not in r3_of(rendered[cond]["system"]):
        pp.append("R3 in %s does not name the supplied number" % cond)

# Neither substitution may leak factor C or frame the absence as a fault.
for phrase in ("smaller of", "horizontal extent", "work it out"):
    if phrase in dims_head.lower():
        pp.append("the dims prompt contains %r, which is factor C" % phrase)
for phrase in ("missing", "error", "should have", "incomplete"):
    if phrase in dims_head.lower():
        pp.append("the dims prompt calls the absence %r, which steers the "
                  "model toward abstaining" % phrase)

# The module's own assertions. The glossary and R3 ones are per condition.
for fn in ("assert_base_states_no_relation", "assert_rungs_isolated"):
    try:
        getattr(P, fn)()
        print("PASS  prompts.%s" % fn)
    except Exception as exc:
        pp.append("prompts.%s: %s" % (fn, exc))
for fn in ("assert_glossary_matches_state", "assert_r3_matches_state"):
    for cond in CONDITIONS:
        try:
            getattr(P, fn)(cond)
            print("PASS  prompts.%s(%s)" % (fn, cond))
        except Exception as exc:
            pp.append("prompts.%s(%s): %s" % (fn, cond, exc))

# The registry, the answer schema and the perception probe must speak ONE
# language, or a reported face can never equal the face that was shown.
if L.block_faces() != frozenset(P.RESTING_FACES):
    pp.append("registry faces %s but the schema lists %s"
              % (sorted(L.block_faces()), sorted(P.RESTING_FACES)))

if pp:
    raise AssertionError("PROMPT CHECK FAILED:\n  " + "\n  ".join(pp))
print()
print("PASS  dims withholds BOTH fields from the state.")
print("PASS  the field list stops promising them, and R3 is substituted")
print("      rather than deleted: the capability check survives, the")
print("      pointer to a supplied number does not.")
print("PASS  neither substitution states the relation or calls the absence")
print("      an error.")
print("PASS  N0 carries no factor wording: %r" % P.RUNGS[RUNG]["text"])

## Cell 5. Cue validation

**Makes model calls.** The two-way perception probe, on a sample of positions
across both faces. Chance is one in two.

If the two faces are not separable above chance, the rest of this notebook
should not be run: every later number is a claim about which source the model
believed, and that claim is empty if the image carries no cue.

This probe was three-way until 2026-08-27. The third option, the middle face,
was withdrawn because no model separated it from `large_face` — GPT scored 58%
on that pair, Fisher p = 0.76, over 81 answered trials. The three-way runs are
kept in `runs/ex2_q1_cue_*.jsonl` as the evidence for that decision.

`mancheck.check_id` is `seq|view|model` and carries **no repeat field**, so
repeats go to separate files rather than colliding on resume.

In [ ]:
# --- Cell 5. Cue validation. MAKES MODEL CALLS. -----------------------------
# BOTH BANKS. This was USABLE[:10], which is every one of them east,
# because "e" sorts before "w". The banks differ in which UR is idle and in
# how the block sits relative to the camera, and this probe is the gate that
# licenses every paid cell below it, so it should not rest on half the
# workspace. Ten east and ten west: the east ten are the ones already
# collected, so the runners resume and only the west are new.
CUE_POSITIONS = ([p for p in USABLE if p.startswith("e")][:10] +
                 [p for p in USABLE if p.startswith("w")][:10])
CUE_REPEATS = 3

# The filename NAMES THE VOCABULARY, and that is not decoration. mancheck's
# check_id is seq|view|model with no vocabulary in it, so a two-way run
# pointed at a three-way file would find every id already present, make no
# calls, and report the old answers as new. Every id in the retired
# ex2_q1_cue_*.jsonl collides exactly that way. mancheck.assert_same_probe
# refuses it, and this keeps the two sets of runs apart in the first place.
CUE_FILE = "ex2_q1_cue%dway_%%s_r%%d.jsonl" % len(FACES)

cue_seqs = [s["seq"] for pos in CUE_POSITIONS for s in by_pos[pos].values()]
n_calls = len(cue_seqs) * len(MODELS) * CUE_REPEATS
print("COST: %d positions x %d faces x %d models x %d repeats = %d calls"
      % (len(CUE_POSITIONS), len(FACES), len(MODELS), CUE_REPEATS, n_calls))
print("chance is %.1f%%: a %d-way forced choice." % (MC.CHANCE, len(FACES)))
# What this run will ACTUALLY cost, which is not the number above once the
# east half is already on disk. CONFIRM_SPEND is still the full design size,
# because that is what the cell is asking permission for; this line is what
# says how much of it has been bought already.
_have = sum(answered(RUNS / (CUE_FILE % (m, r)))
            for m in MODELS for r in range(1, CUE_REPEATS + 1))
print("already answered: %d across %d files, so this run adds about %d calls"
      % (_have, len(MODELS) * CUE_REPEATS, max(0, n_calls - _have)))

CONFIRM_SPEND = None            # <-- set to the number in the COST line

# No out_path: this cell resumes per model and per repeat inside its own
# loop below, so a single file count would not describe it.
if spend_gate(n_calls, CONFIRM_SPEND,
              factors=(("positions", len(CUE_POSITIONS)),
                       ("faces", len(FACES)), ("models", len(MODELS)),
                       ("repeats", CUE_REPEATS))):
    for model in MODELS:
        for rep in range(1, CUE_REPEATS + 1):
            out = RUNS / (CUE_FILE % (model, rep))
            done = answered(out)
            if done >= len(cue_seqs):
                print("skip %s (%d answered already)" % (out.name, done))
                continue
            print("running %s ..." % out.name)
            MC.run(str(CAPTURES), out_path=str(out), model=model,
                   views=("ex2_cam",), seqs=set(cue_seqs))
    print("\ncue validation complete")

### Cue validation results

Accuracy per face with Wilson intervals, and the `small_face` / `large_face`
confusion. A verdict, not a table to be interpreted later.

Two-way since 2026-08-27. The probe was three-way and its hard pair was
`edge` against `large_face`; that face was withdrawn because no model read
it. What that costs is printed with the verdict.

In [ ]:
# --- Cell 5b. Cue validation results. No model calls. -----------------------
cue_rows = []
for model in MODELS:
    for rep in range(1, CUE_REPEATS + 1):
        f = RUNS / (CUE_FILE % (model, rep))
        if f.exists():
            for line in open(f):
                if line.strip():
                    r = json.loads(line)
                    r["model"], r["repeat"] = model, rep
                    cue_rows.append(r)

# Rows from a probe that offered a different set of answers are not the
# same measurement and must never be pooled with these. The retired
# three-way runs sit in the same directory under ex2_q1_cue_*.jsonl and
# would otherwise be read as if they answered this question: restricted to
# the two surviving faces they score 56 percent, because the model could
# still say "edge", and the gate below would read NOT SEPARABLE and shut
# the notebook for the wrong reason.
_foreign = [r for r in cue_rows
            if (r.get("true_face") or r.get("true_pose")) not in FACES
            or (r.get("answer") is not None and r["answer"] not in FACES)]
if _foreign:
    raise AssertionError(
        "%d of %d cue rows answer a different question (e.g. %r). They are "
        "evidence, not input: read them from their own files. %s holds "
        "only this probe's runs."
        % (len(_foreign), len(cue_rows),
           _foreign[0].get("answer") or _foreign[0].get("true_face"),
           CUE_FILE.replace("%s", "<model>").replace("%d", "<rep>")))

if not cue_rows:
    print("no cue files matching %s yet; run cell 5 first"
          % CUE_FILE.replace("%s", "<model>").replace("%d", "<rep>"))
else:
    CHANCE = 100.0 / len(FACES)
    acc_rows, conf_rows = [], []
    for model in MODELS:
        for face in FACES:
            sub = [r for r in cue_rows if r["model"] == model
                   and (r.get("true_face") or r["true_pose"]) == face
                   and r.get("answer") is not None]
            k = sum(1 for r in sub if r["correct"])
            lo, hi = wilson(k, len(sub))
            acc_rows.append([model, face, len({r["seq"] for r in sub}), k,
                             "%.1f" % (100.0 * k / len(sub)) if sub else "NA",
                             "%.1f" % lo if sub else "NA",
                             "%.1f" % hi if sub else "NA"])
        for tf in FACES:
            for pf in FACES:
                n = sum(1 for r in cue_rows if r["model"] == model
                        and (r.get("true_face") or r["true_pose"]) == tf
                        and r.get("answer") == pf)
                conf_rows.append([model, tf, pf, n])

    show(["model", "face", "images", "correct", "acc%", "lo", "hi"], acc_rows)
    print("\nchance is %.1f%% (%d-way forced choice)" % (CHANCE, len(FACES)))
    write_csv("tab_ex2_q1_cue.csv",
              ["model", "resting_face", "n_images", "correct_n",
               "accuracy_pct", "wilson_lo", "wilson_hi"], acc_rows)
    write_csv("tab_ex2_q1_cue_confusion.csv",
              ["model", "true_face", "predicted_face", "n"], conf_rows)

    # Until 2026-08-27 this block restricted the three-way probe to its
    # hard pair, edge against large_face. With two faces the probe IS that
    # pair, so the restriction is gone and the whole accuracy is the
    # verdict. The Wilson bound, not the point estimate, is what decides:
    # a lower bound above chance is the claim that survives a small n.
    print()
    print("=" * 70)
    print("THE DIAGNOSTIC: small_face against large_face")
    print("=" * 70)
    verdict_ok = True
    for model in MODELS:
        sub = [r for r in cue_rows if r["model"] == model
               and (r.get("true_face") or r["true_pose"]) in FACES
               and r.get("answer") is not None]
        k = sum(1 for r in sub
                if r["answer"] == (r.get("true_face") or r["true_pose"]))
        lo, hi = wilson(k, len(sub))
        ok = lo > CHANCE
        verdict_ok &= ok
        print("  %-8s %d/%d correct = %.1f%% [%.1f, %.1f]   %s"
              % (model, k, len(sub), 100.0 * k / len(sub) if sub else float("nan"),
                 lo, hi, "separable" if ok else "NOT SEPARABLE"))
    print()
    if verdict_ok:
        print("VERDICT  the two faces are separable above chance.")
        print("         Q1's contrast is measurable. Proceed.")
    else:
        print("VERDICT  the two faces are NOT separable.")
        print("         Q1's contrast is DEAD: a null on it would measure")
        print("         the render, not the model. Do not run cells 6 and 7.")
        print("         Report this as an instrument result.")
    print()
    print("  This probe cannot distinguish a model that derives the opening")
    print("  from geometry from one that reads posture and applies a rule:")
    print("  small_face stands and large_face lies, so posture alone scores")
    print("  here. The face that separated those readings was withdrawn on")
    print("  2026-08-27 because no model could see it. State this as a")
    print("  limitation wherever the Q1 verdict is reported.")

## Cells 6 and 7. The two conditions at N0

**Make model calls.** Both models, Franka preference, three repeats.
`solo.run` resumes into its output file and skips trials already answered, so
re-running a cell costs nothing and destroys nothing.

The scene count comes from `CALL_SCENES`, set in cell 3. By default that is
**every captured scene**, not just the usable positions: the excluded ones are
asked and then dropped in cell 8 at analysis time, which keeps the exclusion
visible in the data and shows it predates any accuracy result. With 34
captured positions and 32 usable that is 68 scenes rather than 64. Set
`RUN_ALL_POSITIONS = False` in cell 3 to pay only for the usable ones; the
analysis is identical either way.

The count was hardcoded as 90 until 2026-08-27 and had been wrong since the
capture set grew. It is derived now.

In [ ]:
# --- Cell 6. Congruent, N0. MAKES MODEL CALLS. ------------------------------
CONGRUENT_OUT = RUNS / "ex2_q1_congruent_N0.jsonl"

# CALL_SCENES, not scenes: cell 3 decides which positions are paid for, so
# this cell and cell 7 cannot drift apart. The breakdown is printed because
# "68 scenes" against "32 usable positions" looks like a bug otherwise, and
# that question is worth answering before the money is spent, not after.
n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS
_extra = len(CALL_SCENES) - len(USABLE) * len(FACES)
print("COST: %d scenes x %d models x %d repeats = %d calls"
      % (len(CALL_SCENES), len(MODELS), REPEATS, n_calls))
print("      %d usable positions x %d faces = %d scenes%s"
      % (len(USABLE), len(FACES), len(USABLE) * len(FACES),
         ", plus %d at excluded positions, asked so the exclusion is "
         "visible in the data and dropped in cell 8" % _extra
         if _extra else ""))

CONFIRM_SPEND = None            # <-- set to the number in the COST line

# factors= is the guard against a later cell rebinding REPEATS: the gate
# refuses when the counts stop multiplying to the number being confirmed,
# rather than the run quietly coming out a third of the size.
if spend_gate(n_calls, CONFIRM_SPEND, CONGRUENT_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", REPEATS))):
    S.run(str(CAPTURES), out_path=str(CONGRUENT_OUT), models=MODELS,
          conditions=("congruent",), preferences=(PREFERENCE,),
          rungs=(RUNG,), modalities=("V",), kind="pair", repeats=REPEATS)
    print("answered now:", answered(CONGRUENT_OUT))

In [ ]:
# --- Cell 7. Dims, N0. MAKES MODEL CALLS. -----------------------------------
DIMS_OUT = RUNS / "ex2_q1_dims_N0.jsonl"

# CALL_SCENES, not scenes: cell 3 decides which positions are paid for, so
# this cell and cell 7 cannot drift apart. The breakdown is printed because
# "68 scenes" against "32 usable positions" looks like a bug otherwise, and
# that question is worth answering before the money is spent, not after.
n_calls = len(CALL_SCENES) * len(MODELS) * REPEATS
_extra = len(CALL_SCENES) - len(USABLE) * len(FACES)
print("COST: %d scenes x %d models x %d repeats = %d calls"
      % (len(CALL_SCENES), len(MODELS), REPEATS, n_calls))
print("      %d usable positions x %d faces = %d scenes%s"
      % (len(USABLE), len(FACES), len(USABLE) * len(FACES),
         ", plus %d at excluded positions, asked so the exclusion is "
         "visible in the data and dropped in cell 8" % _extra
         if _extra else ""))

CONFIRM_SPEND = None            # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND, DIMS_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", REPEATS))):
    S.run(str(CAPTURES), out_path=str(DIMS_OUT), models=MODELS,
          conditions=("dims",), preferences=(PREFERENCE,),
          rungs=(RUNG,), modalities=("V",), kind="pair", repeats=REPEATS)
    print("answered now:", answered(DIMS_OUT))

## Cell 7b. Dims at N0, with no image

**Makes model calls.** The same condition, sample, models and repeat count as
cell 7, with the picture withheld. This is the floor the dims result has to be
read against: whatever a model gets right with no image at all is what the
structured text alone supports.

Under `dims` the two resting faces are **indistinguishable in text**, so the
contrast measured here is zero by construction and what the cell actually
records is how far a model's answer moves when nothing it can see has moved.
The read-out is cell 15, at the end, so that it can reuse cell 8's loader and
cell 9's definition of Franka share rather than keeping a second copy that
could drift from them.

In [ ]:
# --- Cell 7b. Dims at N0, NO IMAGE. MAKES MODEL CALLS. ----------------------
NOIMAGE_OUT = RUNS / "ex2_q1_dims_N0_noimage.jsonl"

# THE FLOOR FOR CELL 7. Modality "A" renders the same state and attaches no
# image. Whatever a model gets right here is what the structured text alone
# supports, so cell 10's dims contrast has to be read against it: a contrast
# that survives with no picture was never evidence that the model looked.
#
# THE TWO FACES ARE INDISTINGUISHABLE HERE, BY CONSTRUCTION. dims withholds
# resting_face and opening_needed_m, and size_upright_m is quoted in the
# standing frame whichever way the block actually rests, so at one position
# the only difference between the small_face prompt and the large_face
# prompt is the queued task's id. Checked by rendering both and diffing
# them, not assumed. The expected contrast is therefore ZERO and this cell
# measures how much an answer moves when nothing the model can see moves.
# That is the number cell 10's dims contrast has to be bigger than.
#
# BOTH FACES AND THREE REPEATS ANYWAY, rather than half the calls. Grading,
# legality and the Franka share are keyed on the TRUE pose, which the frozen
# state carries whether or not the text mentions it, so asking under both
# labels is what makes this floor comparable cell for cell with cell 7. Half
# the calls would give a floor computed over a different denominator than
# the number it is a floor for.
#
# ITS OWN FILE. The modality is part of the trial_id, so these rows could
# not collide with cell 7's even inside one file. They are still kept apart,
# because cell 8 reads DIMS_OUT whole: a text-only row landing there would
# be folded into the vision condition and would move every table below
# without appearing anywhere as a decision.
#
# REPEATS IS BOUND LOCALLY, not inherited. Cell 7 rebinds REPEATS for its
# own top-up, so a bare REPEATS here would collect whatever the last cell to
# run happened to leave behind, which is the one way this cell could quietly
# under-sample.
NOIMAGE_REPEATS = 3

n_calls = len(CALL_SCENES) * len(MODELS) * NOIMAGE_REPEATS
print("COST: %d scenes x %d models x %d repeats = %d calls"
      % (len(CALL_SCENES), len(MODELS), NOIMAGE_REPEATS, n_calls))
print("      %d usable positions x %d faces = %d scenes, plus the excluded"
      % (len(USABLE), len(FACES), len(USABLE) * len(FACES)))
print("      ones, asked for cell 7's reason and dropped in cell 15.")
print("      No image is attached, so these are the cheapest calls in the")
print("      notebook per trial. solo.cost_table reports what they cost.")

CONFIRM_SPEND = None           # <-- set to the number in the COST line

if spend_gate(n_calls, CONFIRM_SPEND, NOIMAGE_OUT,
              factors=(("scenes", len(CALL_SCENES)), ("models", len(MODELS)),
                       ("repeats", NOIMAGE_REPEATS))):
    S.run(str(CAPTURES), out_path=str(NOIMAGE_OUT), models=MODELS,
          conditions=("dims",), preferences=(PREFERENCE,),
          rungs=(RUNG,), modalities=("A",), kind="pair",
          repeats=NOIMAGE_REPEATS)
    print("answered now:", answered(NOIMAGE_OUT))

## Cell 8. Load and validate replies

No model calls. Nothing is silently dropped: every exclusion is counted and
named, and the rows are kept.

In [ ]:
# --- Cell 8. Load and validate. No model calls. -----------------------------
# load_run FILTERS TO MODELS and reports what it skipped; the reasoning is
# in its docstring, in analysis/ex2/ex2_q_common.py, because cell 15 applies
# the same rules to the no-image rows and two copies would drift.
_c_rows, _c_skip = load_run(CONGRUENT_OUT, "congruent", MODELS)
_d_rows, _d_skip = load_run(DIMS_OUT, "dims", MODELS)
ROWS = _c_rows + _d_rows
SKIPPED_MODELS = _c_skip + _d_skip
print("distinct trials loaded: %d  (models %s)"
      % (len(ROWS), ", ".join(MODELS)))
if SKIPPED_MODELS:
    print("not in the design, left in the files and not counted below: %s"
          % ", ".join("%s %d" % (m, n) for m, n in sorted(SKIPPED_MODELS.items())))

flags = collections.Counter()
for r in ROWS:
    if r.get("error"):
        flags["transport error"] += 1
    if r.get("outcome") == "unparseable":
        flags["unparseable reply"] += 1
    if r.get("arm") and r["arm"] not in (r.get("legal_true") or []):
        flags["named an arm the validator rejects"] += 1
    if not r.get("arm"):
        flags["declined (no arm named)"] += 1
    op, arm = r.get("opening_needed_m"), r.get("arm")
    if op is not None and arm:
        cap = C.ARM_TYPES[C.ARMS[arm]["type"]]["max_grasp_m"] if arm in C.ARMS else None
        if cap is not None and op > cap + 1e-9:
            flags["reported an opening its own arm cannot span"] += 1
    if op is None and r.get("arm"):
        flags["named an arm but reported no opening"] += 1

show(["issue", "n"], [[k, v] for k, v in sorted(flags.items())] or [["none", 0]])
print()
print("Nothing above is dropped. The analysis excludes declines from the")
print("Franka-share denominator (cell 9) and reports them in cell 11; every")
print("other flag is carried through so it can be inspected.")

# The same three exclusions cell 15 applies to the no-image rows, from one
# definition. A decline SURVIVES: it is cell 11's numerator, and it is
# excluded from cell 9's denominator there rather than here.
ANALYSED = keep_analysable(ROWS, USABLE)
print()
print("rows after removing errors and unparseables and restricting to the")
print("%d usable positions: %d" % (len(USABLE), len(ANALYSED)))
print("expected: %d positions x %d faces x %d conditions x %d models x %d reps"
      " = %d" % (len(USABLE), len(FACES), len(CONDITIONS), len(MODELS),
                 REPEATS, len(USABLE) * len(FACES) * len(CONDITIONS)
                 * len(MODELS) * REPEATS))

## Cell 9. Franka share by orientation

Table 2. Declines are excluded from both numerator and denominator; they are
reported separately in cell 11. Wilson bounds are on the proposal denominator.

In [ ]:
# --- Cell 9. Franka share by orientation. No model calls. -------------------
share_rows = []
for cond in CONDITIONS:
    for model in MODELS:
        for face in FACES:
            sub = [r for r in ANALYSED if r["condition"] == cond
                   and r["model"] == model and r["face"] == face]
            k, n = share_counts(sub)     # n is PROPOSALS, not trials
            lo, hi = wilson(k, n)
            share_rows.append([
                cond, model, face, len({r["position"] for r in sub}), n, k,
                "%.1f" % (100.0 * k / n) if n else "NA",
                "%.1f" % lo if n else "NA", "%.1f" % hi if n else "NA"])

show(["condition", "model", "face", "pos", "proposals", "franka", "share%",
      "lo", "hi"], share_rows)
print()
print("A deriving model reads HIGH on small_face and LOW on large_face.")
print("A model using a posture association reads HIGH, LOW, LOW: it separates")
print("standing from flat but not the two flat faces.")
write_csv("tab_ex2_q1_share.csv",
          ["condition", "model", "resting_face", "n_positions", "n_proposals",
           "franka_n", "franka_share_pct", "wilson_lo", "wilson_hi"],
          share_rows)

## Cell 10. Paired contrasts

Table 3, and the by-position companion so the pairing is inspectable.

**On the interval.** The design document asks for Newcombe. Newcombe is an
interval on the difference of two *independent* proportions; these contrasts
are computed within position and then averaged, so the unit is the position and
the correct interval is a t interval over the 29 paired differences. Both are
emitted: `paired_lo`/`paired_hi` is the one to quote, `newcombe_lo`/
`newcombe_hi` is the unpaired comparison the spec named. They are reported side
by side rather than silently substituted.

In [ ]:
# --- Cell 10. Paired contrasts. No model calls. -----------------------------
# ONE contrast. It was two until 2026-08-27, the second being
# edge_minus_large, which is what made the design diagnostic: edge and
# large_face are both flat, so a difference between them could only come
# from geometry. The middle face was withdrawn because no model read it,
# and this is where that loss lands.
CONTRASTS = (("small_minus_large", "small_face", "large_face"),)

bypos_rows, contrast_rows = [], []
ratios = {}
for cond in CONDITIONS:
    for model in MODELS:
        base = [r for r in ANALYSED if r["condition"] == cond
                and r["model"] == model]
        for name, a, b in CONTRASTS:
            # One walk over USABLE feeds both the by-position table and the
            # interval, in one order, so the two cannot disagree about which
            # position is which.
            pairs = paired_diffs(base, USABLE, a, b)
            diffs = [d for _, d in pairs]
            bypos_rows += [[cond, model, pos, name, fmt(d)] for pos, d in pairs]
            mean, plo, phi, npos = paired_mean_ci(diffs)

            # The unpaired comparison the spec asked for, pooled over proposals.
            ka, na = share_counts([r for r in base if r["face"] == a])
            kb, nb = share_counts([r for r in base if r["face"] == b])
            nlo, nhi = newcombe(ka, na, kb, nb)
            contrast_rows.append([cond, model, name, npos,
                                  "%.1f" % mean if mean == mean else "NA",
                                  "%.1f" % nlo if nlo == nlo else "NA",
                                  "%.1f" % nhi if nhi == nhi else "NA",
                                  "%.1f" % plo if plo == plo else "NA",
                                  "%.1f" % phi if phi == phi else "NA",
                                  spans_zero(plo, phi), "NA"])
            ratios[(cond, model, name)] = mean

# dims / congruent, for the one remaining contrast.
for row in contrast_rows:
    cond, model, name = row[0], row[1], row[2]
    if cond == "dims" and name == "small_minus_large":
        num = ratios.get(("dims", model, name))
        den = ratios.get(("congruent", model, name))
        # index 10 is the ratio column; 9 is spans_zero. Counted, not guessed:
        # condition, model, contrast, npos, mean, newc_lo, newc_hi,
        # paired_lo, paired_hi, spans_zero, ratio_to_congruent.
        row[10] = ("%.2f" % (num / den)) if (den is not None and den == den
                                             and abs(den) > 1e-9
                                             and num is not None and num == num
                                             ) else "NA"

show(["condition", "model", "contrast", "npos", "mean", "newc_lo", "newc_hi",
      "paired_lo", "paired_hi", "spans0", "ratio"], contrast_rows)
write_csv("tab_ex2_q1_contrasts.csv",
          ["condition", "model", "contrast", "n_positions", "mean_diff_pts",
           "newcombe_lo", "newcombe_hi", "paired_lo", "paired_hi",
           "spans_zero", "ratio_to_congruent"], contrast_rows)
write_csv("tab_ex2_q1_contrasts_bypos.csv",
          ["condition", "model", "position_id", "contrast", "diff_pts"],
          bypos_rows)

# --- the verdict, against the four patterns in the design document ----------
print()
print("=" * 70)
print("READING, per model, in DIMS")
print("=" * 70)
# THREE patterns, not four. The fourth read "separates standing from flat
# but not the two flat faces: a coarse association, not a derivation", and
# it was the one the design existed to detect. It needed edge_minus_large,
# and the middle face was withdrawn on 2026-08-27 because no model could
# see it. That pattern is now UNTESTABLE, not absent: a model matching it
# reads here as "obtains the opening from the geometry", which is the
# reading it was built to rule out.
#
# Do not let that sit implicitly in the code. It is printed below every
# verdict and belongs in the Limitations section, with runs/ex2_q1_cue_*
# as the evidence that the pattern was real.
for model in MODELS:
    d_sm = ratios.get(("dims", model, "small_minus_large"))
    row = [r for r in contrast_rows if r[0] == "dims" and r[1] == model]
    sm_zero = [r for r in row if r[2] == "small_minus_large"][0][9]
    crow = [r for r in contrast_rows if r[0] == "congruent" and r[1] == model]
    c_zero = [r for r in crow if r[2] == "small_minus_large"][0][9]
    # Branch on the POSITION COUNT, not on spans_zero. spans_zero(nan, nan)
    # is True by design, so a model with no rows at all used to fall through
    # to "cannot obtain it from the scene" -- a reading manufactured from no
    # data, printed in the same words as a real null.
    npos = [r for r in row if r[2] == "small_minus_large"][0][3]

    if not npos:
        v = "NOT RUN. No position carries this contrast for this model."
    elif not sm_zero:
        v = "obtains the opening from the geometry in the scene"
    elif not c_zero:
        v = ("applies the rule when given the opening but cannot obtain it "
             "from the scene")
    else:
        v = ("cannot apply the rule even when given the opening; every later "
             "result for this model is uninterpretable")
    print("  %-8s small-large %s" %
          (model, "NA" if d_sm != d_sm else "%+.1f" % d_sm))
    # A cell where every position gives the same difference has zero
    # variance, so its t interval collapses to zero width and reads as a
    # precision no sample of 32 supports. Say how many positions flipped
    # instead; that is the quantity with an honest interval on it.
    if npos:
        _d = [d for _, d in paired_diffs(
            [r for r in ANALYSED if r["condition"] == "dims"
             and r["model"] == model], USABLE, "small_face", "large_face")]
        _k, _n = full_flip_count(_d)
        if _k == _n and _n:
            _lo, _hi = wilson(_k, _n)
            print("           saturated: %d of %d positions flipped "
                  "completely, Wilson [%.1f, %.1f]. Quote that, not the "
                  "zero-width t interval." % (_k, _n, _lo, _hi))
    print("           -> %s" % v)
    if not sm_zero:
        print("              CANNOT BE DISTINGUISHED from a model that reads")
        print("              posture and applies a rule. small_face stands")
        print("              and large_face lies, so posture alone produces")
        print("              this result. The contrast that separated the")
        print("              two readings needed a third resting face and")
        print("              was withdrawn: see runs/ex2_q1_cue_*.jsonl.")
        print("              The prompt gives posture a SECOND route to the")
        print("              same answer, independent of the withdrawn face:")
        print("              \"size_upright_m\" names the frame its numbers")
        print("              were taken in, so standing-or-flat fixes the")
        print("              opening and the model never has to work out")
        print("              which two extents are horizontal. Both routes")
        print("              are unavoidable with two faces. Read this")
        print("              verdict as posture-plus-lookup, which geometric")
        print("              derivation would also produce.")

## Cell 11. Wait rate by orientation

Table 4. The denominator here is **all replies including declines**, unlike the
share table. The two differ on purpose.

Picking a UR on `large_face` is a judgement that the object is too wide.
Declining is a judgement that the model cannot tell. Both matter and they are
not the same.

In [ ]:
# --- Cell 11. Wait rate by orientation. No model calls. ---------------------
WAIT_THRESHOLD = 5.0        # percent, below which the table is a sentence

wait_rows = []
for cond in CONDITIONS:
    for model in MODELS:
        for face in FACES:
            sub = [r for r in ANALYSED if r["condition"] == cond
                   and r["model"] == model and r["face"] == face]
            k = sum(1 for r in sub if not r.get("arm"))
            lo, hi = wilson(k, len(sub))
            wait_rows.append([cond, model, face, len(sub), k,
                              "%.1f" % (100.0 * k / len(sub)) if sub else "NA",
                              "%.1f" % lo if sub else "NA",
                              "%.1f" % hi if sub else "NA"])

show(["condition", "model", "face", "trials", "declines", "rate%", "lo", "hi"],
     wait_rows)
write_csv("tab_ex2_q1_waits.csv",
          ["condition", "model", "resting_face", "n_trials", "declines_n",
           "decline_rate_pct", "wilson_lo", "wilson_hi"], wait_rows)

rates = [float(r[5]) for r in wait_rows if r[5] != "NA"]
print()
if rates and max(rates) < WAIT_THRESHOLD:
    print("ALL CELLS BELOW %.0f%%. Replace the table with one sentence:" % WAIT_THRESHOLD)
    print('  "Declines were rare throughout, at most %.1f%% in any cell, so'
          % max(rates))
    print('   Franka share reads at face value."')
else:
    print("Waiting is not background. Read the table: a rate rising on")
    print("large_face means the model registers a problem without resolving")
    print("it; a rate rising across dims means it registered the missing")
    print("fields, which is the calibrated response.")

## Cell 12. Reported opening

Inline, no table file. `opening_needed_m` is a self-report and never a scored
endpoint on its own, but it localises the failure: a wrong face with a correct
derivation from it is a different failure from a correct face with a wrong
derivation.

In [ ]:
# --- Cell 12. Reported opening. No model calls. -----------------------------
TOL = 1e-9
for cond in CONDITIONS:
    print("=" * 70)
    print(cond.upper())
    for model in MODELS:
        for face in FACES:
            sub = [r for r in ANALYSED if r["condition"] == cond
                   and r["model"] == model and r["face"] == face]
            true_open = FACTS[face]["grasp_m"]
            stated = [r for r in sub if r.get("opening_needed_m") is not None]
            right = [r for r in stated
                     if abs(r["opening_needed_m"] - true_open) <= 0.006]
            # A wrong number that nevertheless licenses the arm chosen: the
            # model's action follows its own report even though the report is
            # wrong, which is a different failure from acting against it.
            consistent = 0
            for r in stated:
                arm = r.get("arm")
                if not arm or arm not in C.ARMS:
                    continue
                cap = C.ARM_TYPES[C.ARMS[arm]["type"]]["max_grasp_m"]
                if r["opening_needed_m"] <= cap + 1e-9:
                    consistent += 1
            print("  %-8s %-11s stated %2d/%-2d   correct %2d   "
                  "arm consistent with own report %2d"
                  % (model, face, len(stated), len(sub), len(right), consistent))

# COUPLING, BOTH WAYS. This asked only whether the named arm COULD SPAN the
# reported opening until 2026-08-28. A UR opens to 0.140 and every opening
# in this design is 0.050 or 0.100, so every reply naming a UR passed
# automatically, only Franka choices were ever tested, and it read 100
# percent in every cell. A statistic at ceiling whenever the safe arm is
# chosen cannot tell "the arm follows the report" from "this model always
# picks the wide arm", which is exactly the distinction the sentence under
# it claimed to be making.
#
# Agreement is now two-directional, and the two ways of disagreeing mean
# different things, so they are reported apart rather than summed.
print()
print("=" * 70)
print("COUPLING between the reported opening and the arm chosen")
print("=" * 70)
couple_rows = []
for cond in CONDITIONS:
    for model in MODELS:
        sub = [r for r in ANALYSED if r["condition"] == cond
               and r["model"] == model]
        agree, n, over_reach, over_cautious = coupling(sub, FRANKA_MAX)
        lo, hi = wilson(agree, n)
        couple_rows.append([cond, model, n, agree, fmt(pct(agree, n)),
                            fmt(lo), fmt(hi), over_reach, over_cautious])
show(["condition", "model", "coupled", "agree", "agree%", "lo", "hi",
      "said_wide_chose_franka", "said_narrow_chose_ur"], couple_rows)
print()
print("Read the two disagreement columns, not the percentage alone.")
print("  said_wide_chose_franka   the arm cannot close on the opening the")
print("                           model itself reported. Arithmetic, not")
print("                           judgement; grade.self_contradicted counts")
print("                           the same event on the row.")
print("  said_narrow_chose_ur     nothing is violated, but the arm does not")
print("                           follow the report either: an opening a")
print("                           Franka fits, and no Franka named. The old")
print("                           statistic scored every one of these as")
print("                           agreement.")
print()
print("A model whose arm follows its own reported opening is applying the")
print("rule; where it fails, the failure is in obtaining the opening. A model")
print("whose arm contradicts its own report has reasoning and action coming")
print("apart, which is a different finding.")

## Cell 13. Figure

Franka share by orientation, every condition and every model, with intervals and
a reference line at the level a model indifferent between arm types would
produce.

Written as plotted values plus a self-contained TikZ picture: matplotlib is not
installed in this project's environments, and a TikZ figure stays editable in
the thesis rather than arriving as a raster. The colours are declared at the
top of the `.tex` so they can be swapped for the thesis `includes.tex` names.

In [ ]:
# --- Cell 13. Figure. No model calls. ---------------------------------------
plot_rows = []
for cond in CONDITIONS:
    for model in MODELS:
        for face in FACES:
            m = [r for r in share_rows if r[0] == cond and r[1] == model
                 and r[2] == face]
            if m and m[0][6] != "NA":
                plot_rows.append([cond, model, face, float(m[0][6]),
                                  float(m[0][7]), float(m[0][8])])

fig_csv = FIGURES / "fig_ex2_q1_share.csv"
with open(fig_csv, "w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["condition", "model", "resting_face", "share_pct",
                "wilson_lo", "wilson_hi"])
    for r in plot_rows:
        w.writerow([r[0], r[1], r[2], "%.1f" % r[3], "%.1f" % r[4], "%.1f" % r[5]])
print("wrote", rel(fig_csv))

# --- TikZ, self-contained, no pgfplots --------------------------------------
PANEL_W, PANEL_H, GAP = 5.2, 4.2, 1.4
BAR_W, GROUP_GAP = 0.42, 0.30
# One colour per model, and EVERY model needs its own. These were two
# entries with a .get(..., "q1blue") default until claude was added on
# 2026-08-27, at which point the default would have drawn claude in
# gemini's blue: a legend naming three models over bars showing two
# colours, which misreads as a duplicated series rather than a missing
# definition. The assertion below is what makes that impossible.
COLOURS = {"gpt_hi": "q1teal", "gpt": "q1teal", "gemini": "q1blue",
           "claude_md": "q1amber", "claude": "q1amber"}
_uncoloured = [m for m in MODELS if m not in COLOURS]
if _uncoloured:
    raise AssertionError(
        "no colour defined for %s. Add one to COLOURS and a matching "
        "\\definecolor below; two models sharing a colour makes the figure "
        "wrong in a way that reads as a result." % _uncoloured)
if len({COLOURS[m] for m in MODELS}) != len(MODELS):
    raise AssertionError("two models share a colour: %s"
                         % {m: COLOURS[m] for m in MODELS})

def y(pct):
    return PANEL_H * pct / 100.0

lines = [
    "% Experiment 2, Q1. Franka share by resting face.",
    "% Generated by notebooks/ex2_q1_derivation.ipynb -- do not hand-edit.",
    "% Swap the four colour definitions for the thesis includes.tex names.",
    "\\begin{tikzpicture}[x=1cm,y=1cm,font=\\small]",
    "\\definecolor{q1blue}{RGB}{59,110,165}",
    "\\definecolor{q1teal}{RGB}{62,150,146}",
    "\\definecolor{q1amber}{RGB}{198,124,58}",
    "\\definecolor{q1rule}{RGB}{140,140,140}",
]
for pi, cond in enumerate(CONDITIONS):
    x0 = pi * (PANEL_W + GAP)
    lines += [
        "%% --- panel: %s" % cond,
        "\\draw[q1rule] (%.2f,0) -- (%.2f,0);" % (x0, x0 + PANEL_W),
        "\\draw[q1rule] (%.2f,0) -- (%.2f,%.2f);" % (x0, x0, PANEL_H),
        "\\node[anchor=south] at (%.2f,%.2f) {\\textbf{%s}};"
        % (x0 + PANEL_W / 2.0, PANEL_H + 0.15, cond),
    ]
    for gy in (0, 25, 50, 75, 100):
        lines.append("\\draw[q1rule!35] (%.2f,%.2f) -- (%.2f,%.2f);"
                     % (x0, y(gy), x0 + PANEL_W, y(gy)))
        if pi == 0:
            lines.append("\\node[anchor=east,q1rule] at (%.2f,%.2f) {%d};"
                         % (x0 - 0.1, y(gy), gy))
    # a model indifferent between the two arm types names a Franka half the time
    lines.append("\\draw[q1rule,dashed] (%.2f,%.2f) -- (%.2f,%.2f);"
                 % (x0, y(50), x0 + PANEL_W, y(50)))
    for fi, face in enumerate(FACES):
        cx = x0 + PANEL_W * (fi + 0.5) / len(FACES)
        lines.append("\\node[anchor=north,align=center] at (%.2f,-0.12) "
                     "{\\texttt{%s}};" % (cx, face.replace("_", "\\_")))
        for mi, model in enumerate(MODELS):
            m = [r for r in plot_rows if r[0] == cond and r[1] == model
                 and r[2] == face]
            if not m:
                continue
            share, lo, hi = m[0][3], m[0][4], m[0][5]
            bx = cx + (mi - (len(MODELS) - 1) / 2.0) * (BAR_W + 0.06)
            col = COLOURS[model]
            lines.append("\\fill[%s] (%.2f,0) rectangle (%.2f,%.2f);"
                         % (col, bx - BAR_W / 2, bx + BAR_W / 2, y(share)))
            lines.append("\\draw[q1rule,thick] (%.2f,%.2f) -- (%.2f,%.2f);"
                         % (bx, y(lo), bx, y(hi)))
            lines.append("\\draw[q1rule] (%.2f,%.2f) -- (%.2f,%.2f);"
                         % (bx - 0.08, y(lo), bx + 0.08, y(lo)))
            lines.append("\\draw[q1rule] (%.2f,%.2f) -- (%.2f,%.2f);"
                         % (bx - 0.08, y(hi), bx + 0.08, y(hi)))

legx = (len(CONDITIONS) - 1) * (PANEL_W + GAP) + PANEL_W + 0.35
for mi, model in enumerate(MODELS):
    ly = PANEL_H - 0.4 * mi
    lines.append("\\fill[%s] (%.2f,%.2f) rectangle (%.2f,%.2f);"
                 % (COLOURS[model], legx, ly, legx + 0.3, ly + 0.22))
    lines.append("\\node[anchor=west] at (%.2f,%.2f) {%s};"
                 % (legx + 0.38, ly + 0.11, model))
lines.append("\\node[anchor=west,q1rule] at (%.2f,%.2f) "
             "{\\footnotesize indifferent};" % (legx, y(50)))
lines.append("\\node[rotate=90,anchor=south] at (-0.85,%.2f) "
             "{Franka share (\\%%)};" % (PANEL_H / 2.0))
lines.append("\\end{tikzpicture}")

fig_tex = FIGURES / "fig_ex2_q1_share.tex"
fig_tex.write_text("\n".join(lines) + "\n")
print("wrote", rel(fig_tex), "(%d lines)" % len(lines))
print()
print("Compile inside the thesis with \\input{}. It needs only tikz; the four")
print("\\definecolor lines are local so the picture stands alone, and should be")
print("deleted once includes.tex supplies the palette.")

## Cell 14. Provenance

Every input file with its row count and hash, the prompt version, the model
strings and the date, written beside the tables so any number in the chapter
can be traced back.

In [ ]:
# --- Cell 14. Provenance. No model calls. -----------------------------------
prov = []
today = datetime.date.today().isoformat()

for role, path in (("captures", CAPTURES / "consults.jsonl"),
                   ("congruent", CONGRUENT_OUT),
                   ("dims", DIMS_OUT)):
    if not pathlib.Path(path).exists():
        prov.append([role, rel(path), 0, "MISSING", "", "", today])
        continue
    n, vers, mods = run_meta(path)
    if role == "captures":
        n = sum(1 for l in open(path) if l.strip())
    prov.append([role, rel(path), n, sha256(path),
                 vers or P.EX2_PROMPT_VERSION, mods, today])

for model in MODELS:
    for rep in range(1, CUE_REPEATS + 1):
        f = RUNS / (CUE_FILE % (model, rep))
        if f.exists():
            prov.append(["cue_%s_r%d" % (model, rep), rel(f),
                         sum(1 for l in open(f) if l.strip()), sha256(f),
                         P.EX2_PROMPT_VERSION, model, today])

show(["role", "rows", "sha256", "prompt_version", "models"],
     [[r[0], r[2], (r[3] or "")[:12], r[4], r[5]] for r in prov])
write_csv("tab_ex2_q1_provenance.csv",
          ["role", "path", "rows", "sha256", "prompt_version", "model_string",
           "run_date"], prov)

print()
print("DESIGN FACTS THAT MUST BE DISCLOSED IN THE CHAPTER")
print("-" * 70)
print("1. The idle UR is chosen per position, as the one that can reach the")
print("   object. Every capture was written with ur_w idle, which is right")
print("   for the west positions and wrong for the east: there the object is")
print("   reached by ur_e, so idle-and-reachable collapsed to franka_n alone")
print("   and large_face had NO legal arm. The arm states are set after the")
print("   frames are rendered and no arm is ever commanded to move, so this")
print("   is a text-layer choice that contradicts nothing in the image.")
print("2. %d positions carry the contrast and show the block. %s excluded"
      % (len(USABLE), ", ".join(sorted(excluded)) or "none"))
print("   for legality: the franka cannot reach, so there is no arm choice.")
print("   %s excluded for occlusion: the block is not visible enough to"
      % (", ".join(sorted(OCCLUDED)) or "none"))
print("   judge, measured from pixels alone and blind to any model reply.")
print("3. Only ex2_cam was captured, so there is no viewpoint control.")
print("4. The design used THREE resting faces until 2026-08-27. The third,")
print("   the middle face, gave the only contrast between two orientations")
print("   that were both flat, and so the only test that separated deriving")
print("   the opening from geometry from reading posture and applying a")
print("   rule. It was withdrawn because no model could see it: GPT scored")
print("   58%% on that pair, Fisher p = 0.76, over 81 answered trials, while")
print("   answering a plain standing-or-flat question 18 times out of 18.")
print("   Evidence: runs/ex2_q1_cue_*.jsonl, retained. CONSEQUENCE: every")
print("   'obtains the opening from the geometry' verdict in this notebook")
print("   is consistent with posture-plus-rule and does not exclude it.")
print("5. Posture reaches the same answer by a SECOND route, which retiring")
print("   the third face did not create and no prompt wording removes.")
print("   'size_upright_m' states the frame its three numbers were taken")
print("   in -- standing on the smallest face -- so with two captured")
print("   faces the object is either in that frame or flat, and a two-way")
print("   posture judgement fixes the opening. The model never has to")
print("   work out which two extents are horizontal.")
print("   NOT PATCHED, deliberately. The extents reach the model as an")
print("   unordered set whatever the field is called, so dropping the")
print("   frame from the gloss would not restore a step; it would add an")
print("   ambiguity, since the numbers could then be read as the extents")
print("   AS PLACED and give min(0.100, 0.050) = 0.050 on large_face --")
print("   the wrong opening on the correct condition, which is")
print("   measurement error rather than a harder task.")
print("   Q1 therefore measures posture-plus-lookup and cannot separate")
print("   it from geometric derivation. State that in Limitations.")
print("6. Contrasts are paired within position; the quoted interval is the")
print("   t interval over positions, not Newcombe. Both are in the CSV.")
print("7. Prompt version %s. Rung %s only." % (P.EX2_PROMPT_VERSION, RUNG))

## Cell 15. The no-image floor

No model calls. Reads cell 7b's file and puts it beside the vision result.

Two things are being asked. First, does the dims contrast need the picture:
the floor contrast should be indistinguishable from zero, because the two
prompts differ only in a task id, and a floor that is **not** zero is an
instrument fault rather than a finding. Second, what does a model do when the
opening is genuinely unavailable, since waiting is the defensible answer there
and R3 cannot be satisfied for either arm.

In [ ]:
# --- Cell 15. The no-image floor. No model calls. ---------------------------
# Every helper here is the one cells 8, 9 and 10 use, imported from
# analysis/ex2/ex2_q_common.py. A floor computed by a different rule than
# the number it is a floor for is not a floor, and that is now enforced by
# there being one definition rather than a comment promising there are two.
NOIMAGE_ROWS, NOIMAGE_SKIPPED = load_run(NOIMAGE_OUT, "dims_noimage", MODELS)
NOIMAGE = keep_analysable(NOIMAGE_ROWS, USABLE)
print("no-image rows kept: %d   expected %d positions x %d faces x %d models"
      " x %d reps = %d"
      % (len(NOIMAGE), len(USABLE), len(FACES), len(MODELS), NOIMAGE_REPEATS,
         len(USABLE) * len(FACES) * len(MODELS) * NOIMAGE_REPEATS))
if NOIMAGE_SKIPPED:
    print("not in the design, left in the file and not counted: %s"
          % ", ".join("%s %d" % (m, n)
                      for m, n in sorted(NOIMAGE_SKIPPED.items())))

if not NOIMAGE:
    print()
    print("Cell 7b has not been run, so there is no floor to report and the")
    print("dims contrast in cell 10 stands without one. Nothing below runs.")
else:
    # --- share by face, exactly as cell 9 computes it ------------------------
    floor_share = []
    for model in MODELS:
        for face in FACES:
            sub = [r for r in NOIMAGE
                   if r["model"] == model and r["face"] == face]
            k, n = share_counts(sub)
            lo, hi = wilson(k, n)
            floor_share.append([
                model, face, len({r["position"] for r in sub}), len(sub), n,
                len(sub) - n, k,
                "%.1f" % (100.0 * k / n) if n else "NA",
                "%.1f" % lo if n else "NA", "%.1f" % hi if n else "NA"])

    show(["model", "face", "pos", "trials", "proposals", "declines", "franka",
          "share%", "lo", "hi"], floor_share)
    write_csv("tab_ex2_q1_noimage_share.csv",
              ["model", "resting_face", "n_positions", "n_trials",
               "n_proposals", "declines_n", "franka_n", "franka_share_pct",
               "wilson_lo", "wilson_hi"], floor_share)

    # --- the contrast, paired within position as cell 10 pairs it -----------
    print()
    floor_contrast = []
    for model in MODELS:
        base = [r for r in NOIMAGE if r["model"] == model]
        diffs = [d for _, d in paired_diffs(base, USABLE,
                                            "small_face", "large_face")]
        mean, plo, phi, npos = paired_mean_ci(diffs)
        vis = ratios.get(("dims", model, "small_minus_large"))
        floor_contrast.append([
            model, npos,
            "%.1f" % mean if mean == mean else "NA",
            "%.1f" % plo if plo == plo else "NA",
            "%.1f" % phi if phi == phi else "NA",
            spans_zero(plo, phi),
            "%.1f" % vis if (vis is not None and vis == vis) else "NA"])

    show(["model", "npos", "floor", "paired_lo", "paired_hi", "spans0",
          "with_image"], floor_contrast)
    write_csv("tab_ex2_q1_noimage_contrast.csv",
              ["model", "n_positions", "floor_contrast_pts", "paired_lo",
               "paired_hi", "spans_zero", "vision_contrast_pts"],
              floor_contrast)

    # --- what each model's pair of numbers means ----------------------------
    print()
    print("=" * 70)
    print("READING, per model")
    print("=" * 70)
    for row in floor_contrast:
        model, floor_spans, vis = row[0], row[5], row[6]
        vrow = [r for r in contrast_rows
                if r[0] == "dims" and r[1] == model
                and r[2] == "small_minus_large"]
        vis_spans = vrow[0][9] if vrow else True
        print("  %-9s floor %s   with image %s" % (model, row[2], vis))
        if not floor_spans:
            print("           -> INSTRUMENT FAULT, not a result. The two")
            print("              prompts differ only in a task id, so a")
            print("              contrast here cannot come from the design.")
            print("              Do not quote this model's dims contrast")
            print("              until this is explained.")
        elif not vis_spans:
            print("           -> the contrast needs the picture: nothing")
            print("              without it, an effect with it.")
        else:
            print("           -> null either way. This model shows no")
            print("              contrast with the image and none without,")
            print("              so the image is not what it is missing.")

    # --- declining is the defensible answer here ----------------------------
    print()
    print("=" * 70)
    print("DECLINE RATE with no image")
    print("=" * 70)
    print("R3 cannot be satisfied for either arm here: the opening is not")
    print("stated and there is no picture to obtain it from, so waiting is")
    print("the defensible answer and naming an arm is a guess. A model that")
    print("declines is not failing this cell.")
    for model in MODELS:
        sub = [r for r in NOIMAGE if r["model"] == model]
        d = sum(1 for r in sub if not r.get("arm"))
        lo, hi = wilson(d, len(sub))
        print("  %-9s %3d of %3d declined = %5.1f%% [%.1f, %.1f]"
              % (model, d, len(sub), 100.0 * d / len(sub) if sub else float("nan"),
                 lo, hi))